# 作业 1

## 2 概述、线性代数和 NDArray

### 2.1 理论计算题

已知向量 a = [2, -1, 3]T ，b = [1, 4, -2]T ，矩阵

```
A = [[1, 2, 3],
     [4, 5, 6],
     [7, 8, 9]]

B = [[10, 11],
     [12, 13],
     [14, 15]]
```

计算：

1. 向量点积 a · b；
2. 矩阵乘法 A × B （结果矩阵的形状？）；
3. 向量 a 的 Frobenius 范数。

**解答：**

1. **向量点积 a · b**

   a · b = (2)(1) + (-1)(4) + (3)(-2) = 2 - 4 - 6 = -8

2. **矩阵乘法 A × B**

   A 是 3x3 矩阵，B 是 3x2 矩阵。因此，A × B 的结果将是 3x2 矩阵。

   ```
   A × B = [[76, 82],
            [184, 199],
            [292, 316]]
   ```

3. **向量 a 的 Frobenius 范数**

   ||a||_F = sqrt(2^2 + (-1)^2 + 3^2) = sqrt(14)

### 2.2 编程题

In [ ]:
import numpy as np

# 1. 创建一个形状为 3 × 4 的随机矩阵 X ，元素服从标准正态分布。
X = np.random.randn(3, 4)
print("Matrix X:\n", X)

# 2. 创建一个形状为 4 × 2 的全 1 矩阵 Y。
Y = np.ones((4, 2))
print("Matrix Y:\n", Y)

# 3. 计算矩阵乘法 Z = X × Y。
Z = np.dot(X, Y)
print("Matrix Z (X @ Y):\n", Z)

# 4. 输出 Z 的第一行和第二列交叉处的元素，以及 Z 的第 2 行所有元素。
element_Z_0_1 = Z[0, 1]
print("Element at Z[0, 1]:", element_Z_0_1)

row_Z_1 = Z[1, :]
print("Second row of Z:", row_Z_1)

# 5. 计算 Z 的 Frobenius 范数。
frobenius_norm_Z = np.linalg.norm(Z, 'fro')
print("Frobenius norm of Z:", frobenius_norm_Z)

## 3 文档 2：概率与统计

### 3.1 理论计算题

某疾病在人群中的患病率为 0.1%。现有一种检测方法：

• 若患病，检测呈阳性的概率为 99%（灵敏度）；
• 若未患病，检测呈阳性的概率为 2%（假阳性率）。

一个人检测结果为阳性，求他真正患病的概率（使用贝叶斯公式）。

**解答：**

设 H 为患病，E 为阳性。P(H|E) = [P(E|H) * P(H)] / [P(E|H)P(H) + P(E|H')P(H')]

P(H|E) = (0.99 * 0.001) / (0.99 * 0.001 + 0.02 * 0.999) ≈ 0.04721

### 3.2 编程题

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

n = 10000
m = 1000

means = np.array([np.mean(np.random.uniform(0, 1, n)) for _ in range(m)])

plt.figure(figsize=(10, 6))
plt.hist(means, bins=30, density=True, alpha=0.6, color='g', label='Sample Means')
mu_theoretical = 0.5
sigma_theoretical = np.sqrt((1/12) / n)
x = np.linspace(min(means), max(means), 100)
plt.plot(x, norm.pdf(x, mu_theoretical, sigma_theoretical), 'r-', label='Theoretical')
plt.legend()
plt.show()

print("Actual variance:", np.var(means))

## 4 文档 3：导数、反向传播和复杂度

### 4.1 理论计算题

1. ∂z/∂w₁ = 4(2w₁ + w₂ - 3), ∂z/∂w₂ = 2(2w₁ + w₂ - 3)
2. w₁ = 0.5, w₂ = 1 => 梯度 [-4, -2]

### 4.2 编程题

In [ ]:
import torch
import numpy as np

x1, u1, u2 = 2.0, 1.5, 0.5
a = x1 * u1
b = a + u2
L = b**2

dL_db = 2 * b
dL_du2 = dL_db * 1
dL_du1 = dL_db * 1 * x1

u1_t = torch.tensor(u1, requires_grad=True)
u2_t = torch.tensor(u2, requires_grad=True)
L_t = (x1 * u1_t + u2_t)**2
L_t.backward()

print(f"Manual: du1={dL_du1}, du2={dL_du2}")
print(f"Torch: du1={u1_t.grad}, du2={u2_t.grad}")

## 5 文档 4：线性方法、基础优化和 SOFTMAX 回归

### 5.1 理论计算题

∂L/∂w = -(1/N) Σ (yᵢ - (wᵀxᵢ + b))xᵢ
∂L/∂b = -(1/N) Σ (yᵢ - (wᵀxᵢ + b))

### 5.2 编程题

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

digits = load_digits()
X, y = digits.data, digits.target
y_onehot = OneHotEncoder(sparse_output=False).fit_transform(y.reshape(-1, 1))
X_train, X_test, y_train, y_test = train_test_split(X, y_onehot, test_size=0.2)

def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

W = np.random.randn(X.shape[1], 10) * 0.01
b = np.zeros((1, 10))

for epoch in range(50):
    for i in range(0, X_train.shape[0], 32):
        X_b, y_b = X_train[i:i+32], y_train[i:i+32]
        pred = softmax(np.dot(X_b, W) + b)
        grad = pred - y_b
        W -= 0.1 * np.dot(X_b.T, grad) / len(X_b)
        b -= 0.1 * np.sum(grad, axis=0) / len(X_b)

acc = np.mean(np.argmax(softmax(np.dot(X_test, W) + b), axis=1) == np.argmax(y_test, axis=1))
print(f"Test Accuracy: {acc:.4f}")

## 6 文档 5：最大似然估计和逻辑回归

### 6.1 理论计算题

1. L(μ, σ²) = Π (1/sqrt(2πσ²)) exp(-(xi-μ)²/2σ²)
2. μ_MLE = x̄
3. σ²_MLE = (1/n)Σ(xi-x̄)²

### 6.2 编程题

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X0 = np.random.randn(200, 2) + [-1, -1]
X1 = np.random.randn(200, 2) + [1, 1]
X = np.vstack((X0, X1))
y = np.hstack((np.zeros(200), np.ones(200)))

w, b = np.random.randn(2) * 0.01, 0.0
for _ in range(1000):
    pred = 1 / (1 + np.exp(-(np.dot(X, w) + b)))
    grad = pred - y
    w -= 0.1 * np.dot(X.T, grad) / len(y)
    b -= 0.1 * np.mean(grad)

plt.scatter(X[:,0], X[:,1], c=y)
x_b = np.array([X[:,0].min(), X[:,0].max()])
plt.plot(x_b, (-w[0]*x_b - b)/w[1], 'r--')
plt.show()